# Module 6 · Advanced handling of LLM responses (streaming, batching, structured output)

Run each section top to bottom. Every cell is self-contained, so change the prompt or the numbers and re-run it.

> 🔐 **Keys** — nothing in this notebook contains a key. In Colab open the **🔑 Secrets** panel (left sidebar), click **+ Add new secret**,
> type the *name* exactly as below, paste the *value*, and switch on **Notebook access**. If you skip this, you'll be asked once (hidden input).

| Secret name | What it is | Get it at |
|---|---|---|
| `GEMINI_API_KEY` | Google Gemini API key | https://aistudio.google.com/apikey |

**Never** paste a key into a code cell, commit it to GitHub, or share a notebook that shows it in an output. If one leaks, delete it in the provider's console and create a new one.

In [ ]:
!pip install -q google-genai pydantic langchain-core langchain-google-genai

In [ ]:
import os, getpass

def get_secret(name, *, required=True, hint=""):
    """1) environment  2) Colab Secrets (key icon in the left sidebar)  3) hidden prompt.
    The value is kept in memory / os.environ only - it is never printed or saved in the notebook."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata          # works only inside Colab
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass.getpass(f"Paste {name} {hint}(input is hidden): ").strip()
    if val:
        os.environ[name] = val
    return val

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=get_secret("GEMINI_API_KEY", hint="(from aistudio.google.com/apikey) "))
# Change to "gemini-2.5-flash" if your key does not have access to the newer model.
MODEL = "gemini-3.5-flash-lite"
print('Ready. Model =', MODEL)

## 1 · Normal (blocking) call vs streaming
With a normal call you wait for the whole answer. With streaming the **first words** arrive early — total time is similar, but the *perceived* wait is much shorter.

In [ ]:
import time
PROMPT = "Explain how a bill becomes law in India in about 150 words."

t0 = time.perf_counter()
r = client.models.generate_content(model=MODEL, contents=PROMPT)
print(f"BLOCKING: whole answer after {time.perf_counter()-t0:.2f}s")
print(r.text[:200], "...\n")

t0 = time.perf_counter(); first = None; text = ""
for chunk in client.models.generate_content_stream(model=MODEL, contents=PROMPT):
    if first is None:
        first = time.perf_counter() - t0
    text += chunk.text or ""
print(f"STREAMING: first words after {first:.2f}s, finished after {time.perf_counter()-t0:.2f}s")

## 2 · Async streaming and async batching
`await` lets many requests overlap. Total time ≈ the slowest call, not the sum.

In [ ]:
import asyncio, time

QUESTIONS = ["What is an API?", "What is a token?", "What is a prompt?", "What is temperature?", "What is RAG?"]

async def ask(q):
    r = await client.aio.models.generate_content(model=MODEL, contents=q + " Answer in one sentence.")
    return q, r.text.strip()

async def run_all():
    return await asyncio.gather(*(ask(q) for q in QUESTIONS))

# Colab already runs an event loop, so we simply `await`
t0 = time.perf_counter()
results = await run_all()
print(f"{len(QUESTIONS)} questions concurrently in {time.perf_counter()-t0:.2f}s")
for q, a in results:
    print("-", q, "->", a)

# sequential, for comparison
t0 = time.perf_counter()
for q in QUESTIONS:
    client.models.generate_content(model=MODEL, contents=q + " Answer in one sentence.")
print(f"sequential: {time.perf_counter()-t0:.2f}s")

## 3 · Batching many inputs into ONE prompt
Fewer requests → lower cost and fewer rate-limit hits. Ask for numbered output so you can split it again.

In [ ]:
import json, re
TICKETS = [
    "My order #4821 arrived broken.", "How do I change my password?", "Love the new dark mode!",
    "I was charged twice this month.", "Can you add an export-to-PDF button?", "The app crashes when I upload a photo.",
]
numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(TICKETS))
prompt = f"""Classify each support ticket as complaint, question, praise or suggestion.
Return ONLY a JSON list of objects: [{{"id": 1, "category": "..."}}, ...]

Tickets:
{numbered}"""
print(prompt, "\n")
raw = client.models.generate_content(model=MODEL, contents=prompt).text
print(raw)
data = json.loads(re.search(r"\[.*\]", raw, re.S).group())
for d in data:
    print(d["id"], TICKETS[d["id"]-1][:40].ljust(42), "->", d["category"])

## 4 · Structured output with Pydantic (validate, don't trust)
The model is told the schema; **Pydantic** checks the result. If validation fails, handle it instead of crashing.

In [ ]:
from pydantic import BaseModel, Field, ValidationError
from typing import List

class Resume(BaseModel):
    name: str
    skills: List[str]
    experience: int = Field(..., ge=0, description="Years of experience")

texts = [
    "Rahul has four years of experience in Python and Machine Learning.",
    "Anjali worked on web development with JavaScript and React for 2 years.",
    "Name: Vikram. Skills: C++, Rust. Experience: two years.",   # words instead of a number
]
for t in texts:
    r = client.models.generate_content(
        model=MODEL, contents="Extract the resume fields from: " + t,
        config=types.GenerateContentConfig(response_mime_type="application/json", response_schema=Resume))
    try:
        print("OK  ", Resume.model_validate_json(r.text))
    except ValidationError as e:
        print("FAIL", t, "->", e.errors()[0]["msg"])

## 5 · The same idea with LangChain's `PydanticOutputParser` and `abatch`

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

parser = PydanticOutputParser(pydantic_object=Resume)
prompt = PromptTemplate(
    template="Extract the following fields from the text.\n{format_instructions}\n\nText:\n{text}",
    input_variables=["text"], partial_variables={"format_instructions": parser.get_format_instructions()})
llm = ChatGoogleGenerativeAI(model=MODEL, google_api_key=os.environ["GEMINI_API_KEY"])
chain = prompt | llm | parser

outs = await chain.abatch([{"text": t} for t in texts], return_exceptions=True)
for t, o in zip(texts, outs):
    print(type(o).__name__, "|", o if not isinstance(o, Exception) else str(o)[:120])

## 6 · Mini project — customer-feedback assistant
Classify the feedback **and** draft a polite reply, with a safe fallback if the model's format is wrong.

In [ ]:
def classify_and_respond(feedback: str) -> dict:
    prompt = f"""You are a customer support assistant.
Classify the feedback as exactly one of: complaint, suggestion, praise.
Then write a short polite reply.

Feedback: "{feedback}"

Format:
Category: <one word>
Response: <your reply>"""
    text = client.models.generate_content(model=MODEL, contents=prompt).text.strip().splitlines()
    try:
        return {"category": text[0].split(":", 1)[1].strip(), "response": text[1].split(":", 1)[1].strip()}
    except Exception:
        return {"category": "unknown", "response": "(could not parse the model output)", "raw": text}

for fb in ["I love the service!", "Your app crashes every time I open it.", "Could you add a dark mode to the app?"]:
    print(fb, "->", classify_and_respond(fb), "\n")